# LeetCode 1482 — Minimum Number of Days to Make m Bouquets bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-days-to-make-m-bouquets/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def minDays(self, bloomDay: list[int], m: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Viết hàm kiểm tra `can(day)`: "đến ngày `day` có làm được ít nhất `m` bó, mỗi bó `k` bông **liền kề**?" bằng một lần quét đếm **chuỗi liên tiếp**.
2. Nhận ra tính **đơn điệu**: nếu làm được ở ngày `x` thì làm được ở mọi ngày `y > x` ⇒ **tìm kiếm nhị phân trên đáp án**.
3. Viết tìm kiếm nhị phân dạng "tìm giá trị nhỏ nhất thỏa điều kiện" **không lỗi biên** (`lo < hi`, `hi = mid`, `lo = mid + 1`).
4. Kiểm tra sớm trường hợp **không thể**: `m · k > n` ⇒ `-1`.
5. Biết cách tiếp cận khác: xử lý bông theo thứ tự nở + **hợp nhất đoạn (Union-Find)**, đếm số bó theo thời gian.

## 2. Đề bài & ví dụ

Cho mảng `bloomDay` (độ dài `n`), số `m` và số `k`. Cần làm `m` bó hoa; mỗi bó dùng đúng `k` bông **kề nhau** trong vườn. Bông thứ `i` nở vào ngày `bloomDay[i]` và từ đó dùng được cho **đúng một** bó.

Trả về **số ngày nhỏ nhất** phải chờ để làm được `m` bó; nếu không thể, trả `-1`.

**Ví dụ 1:** `bloomDay = [1,10,3,10,2], m = 3, k = 1` → `3`. Sau ngày 1: `[x,_,_,_,_]` (1 bó); ngày 2: `[x,_,_,_,x]` (2 bó); ngày 3: `[x,_,x,_,x]` (3 bó).

**Ví dụ 2:** `bloomDay = [1,10,3,10,2], m = 3, k = 2` → `-1` (cần 6 bông, chỉ có 5).

**Ví dụ 3:** `bloomDay = [7,7,7,7,12,7,7], m = 2, k = 3` → `12`. Ngày 7: `[x,x,x,x,_,x,x]` chỉ làm được 1 bó (hai bông cuối không đủ 3 liền kề); ngày 12 mọi bông đã nở.

**Ràng buộc:**
- `bloomDay.length == n`, `1 <= n <= 10^5`
- `1 <= bloomDay[i] <= 10^9`
- `1 <= m <= 10^6`, `1 <= k <= n`

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

examples = [
    ([1, 10, 3, 10, 2], 3, 1, 3),
    ([1, 10, 3, 10, 2], 3, 2, -1),
    ([7, 7, 7, 7, 12, 7, 7], 2, 3, 12),
]
for b, m, k, want in examples:
    print(f"bloomDay={b}, m={m}, k={k} -> kỳ vọng {want}")

bloomDay=[1, 10, 3, 10, 2], m=3, k=1 -> kỳ vọng 3
bloomDay=[1, 10, 3, 10, 2], m=3, k=2 -> kỳ vọng -1
bloomDay=[7, 7, 7, 7, 12, 7, 7], m=2, k=3 -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1 Đếm chuỗi liên tiếp thỏa điều kiện

Duyệt mảng, giữ biến `run` = độ dài chuỗi liên tiếp hiện tại thỏa điều kiện; gặp phần tử không thỏa thì `run = 0`. Khi `run` đạt `k` thì "cắt" một bó và **đặt lại `run = 0`** (các bông đó đã dùng).

In [2]:
def bouquets_on(bloomDay, k, day):
    count = run = 0
    for b in bloomDay:
        if b <= day:
            run += 1
            if run == k:
                count += 1
                run = 0
        else:
            run = 0
    return count

b = [7, 7, 7, 7, 12, 7, 7]
for day in (6, 7, 11, 12):
    print(f"ngày {day}: nở = {''.join('x' if x <= day else '_' for x in b)}, số bó (k=3) = {bouquets_on(b, 3, day)}")

ngày 6: nở = _______, số bó (k=3) = 0
ngày 7: nở = xxxx_xx, số bó (k=3) = 1
ngày 11: nở = xxxx_xx, số bó (k=3) = 1
ngày 12: nở = xxxxxxx, số bó (k=3) = 2


### 3.2 Tính đơn điệu

Ngày càng muộn, tập bông đã nở **chỉ lớn thêm**, nên số bó làm được **không giảm**. Vì thế điều kiện `can(day) = (bouquets_on(day) >= m)` có dạng `F F F … F T T T … T` theo `day`.

In [3]:
b, m, k = [1, 10, 3, 10, 2], 3, 1
print([(d, bouquets_on(b, k, d) >= m) for d in range(1, 11)])

[(1, False), (2, False), (3, True), (4, True), (5, True), (6, True), (7, True), (8, True), (9, True), (10, True)]


### 3.3 Tìm kiếm nhị phân "giá trị nhỏ nhất thỏa điều kiện"

Trên đoạn `[lo, hi]` mà `hi` chắc chắn thỏa:

```python
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid        # mid có thể là đáp án, giữ lại
    else:       lo = mid + 1    # mid chắc chắn không phải
return lo
```

Mỗi vòng thu hẹp đoạn ít nhất 1 và luôn giữ đáp án bên trong ⇒ dừng sau `O(log(hi − lo))` vòng.

In [4]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(0, 100, lambda x: x * x >= 50))     # số nhỏ nhất có bình phương >= 50
print(first_true(1, 10**9, lambda x: x >= 123456789), "| số vòng ≈ log2(1e9) ≈ 30")

8
123456789 | số vòng ≈ log2(1e9) ≈ 30


### 3.4 `bisect_left` trên mảng đã sắp

`bisect_left(a, x)` trả vị trí đầu tiên `i` với `a[i] >= x`. Hữu ích khi tìm nhị phân trên **danh sách các ngày phân biệt đã sắp** thay vì trên mọi số nguyên.

In [5]:
days = sorted(set([1, 10, 3, 10, 2]))
print(days, bisect_left(days, 3), bisect_left(days, 4))

[1, 2, 3, 10] 2 3


### 3.5 Union-Find (Disjoint Set Union) tối giản

Mỗi phần tử trỏ tới "cha"; `find(x)` đi lên tới gốc (kèm nén đường đi); `union(a, b)` nối hai gốc. Ta dùng nó để ghép các bông **đã nở liền kề** thành đoạn và theo dõi độ dài đoạn.

In [6]:
parent = list(range(6))
size = [1] * 6
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return ra
    if size[ra] < size[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    size[ra] += size[rb]
    return ra
union(0, 1); union(1, 2); union(4, 5)
print("gốc:", [find(i) for i in range(6)], "| kích thước đoạn chứa 2:", size[find(2)])

gốc: [0, 0, 0, 3, 4, 4] | kích thước đoạn chứa 2: 3


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `bloomDay = [7,7,7,7,12,7,7]`, `m = 2`, `k = 3`.

| ngày | trạng thái vườn | chuỗi liền kề | số bó (k=3) | đủ m=2? |
|---|---|---|---|---|
| 6 | `_ _ _ _ _ _ _` | – | 0 | ✘ |
| 7 | `x x x x _ x x` | 4, 2 | 1 + 0 = 1 | ✘ |
| 8..11 | như ngày 7 | 4, 2 | 1 | ✘ |
| 12 | `x x x x x x x` | 7 | 2 | ✔ |

Hai nhận xét:
1. Số bó ở một ngày = tổng `⌊độ dài chuỗi / k⌋` trên các chuỗi bông đã nở liền kề.
2. Đáp án luôn là **một trong các giá trị của `bloomDay`** (số bó chỉ thay đổi đúng vào ngày có bông nở).

In [7]:
b, m, k = [7, 7, 7, 7, 12, 7, 7], 2, 3
for day in sorted(set(b)):
    runs, cur = [], 0
    for x in b + [float("inf")]:
        if x <= day:
            cur += 1
        else:
            if cur:
                runs.append(cur)
            cur = 0
    print(f"ngày {day}: chuỗi {runs} -> số bó = {sum(r // k for r in runs)}")

ngày 7: chuỗi [4, 2] -> số bó = 1
ngày 12: chuỗi [7] -> số bó = 2


## 5. Cách 1 — Brute Force / baseline: thử từng ngày ứng viên

**Ý tưởng:** nếu `m · k > n` thì `-1`. Ngược lại, thử các ngày phân biệt trong `bloomDay` theo thứ tự tăng; ngày đầu tiên làm đủ `m` bó là đáp án.

Đúng vì đáp án là một giá trị của `bloomDay` (mục 4) và tính đơn điệu.

In [8]:
def min_days_brute(bloomDay, m, k):
    if m * k > len(bloomDay):
        return -1
    for day in sorted(set(bloomDay)):
        if bouquets_on(bloomDay, k, day) >= m:
            return day
    return -1          # không xảy ra khi m*k <= n (ngày lớn nhất luôn đủ)

for b, m, k, want in examples:
    got = min_days_brute(b, m, k)
    print(got, want, "✔" if got == want else "✘")

3 3 ✔
-1 -1 ✔
12 12 ✔


**Độ phức tạp:** `D` ngày phân biệt (≤ `n`), mỗi lần kiểm tra `O(n)` ⇒ `O(n · D)` = `O(n²)` xấu nhất = `10^10`. (Thử **mọi** số nguyên từ 1 tới `max(bloomDay)` còn tệ hơn: tới `10^9` ngày.)

**Nút thắt:** ta kiểm tra tuần tự từng ngày, trong khi điều kiện là **đơn điệu** — có thể nhảy cóc bằng chia đôi.

In [9]:
rng = random.Random(1482)
for n in (1000, 2000, 4000):
    b = rng.sample(range(1, 10**9), n)             # mọi ngày khác nhau
    t = time.perf_counter()
    r = min_days_brute(b, n // 4, 3)               # cần nhiều bông -> ngày muộn
    print(f"n={n}: đáp án={r}, {time.perf_counter() - t:.3f}s")

n=1000: đáp án=878518383, 0.035s


n=2000: đáp án=859958051, 0.141s


n=4000: đáp án=861891588, 0.618s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** hàm `can(day)` **đơn điệu** (F…F T…T) ⇒ dùng **tìm kiếm nhị phân trên đáp án**: thay vì hỏi "đáp án là bao nhiêu?", hỏi "ngày `mid` có đủ không?" và chia đôi khoảng.

**Quy tắc:**
- **Nếu** `m · k > n` **thì** trả `-1` ngay (không đủ bông dù tất cả đều nở).
- Khoảng tìm: `lo = min(bloomDay)`, `hi = max(bloomDay)` (ở `hi` mọi bông đã nở ⇒ chắc chắn đủ).
- **Nếu** `can(mid)` **thì** `hi = mid` (mid có thể là đáp án) **ngược lại** `lo = mid + 1`.

**Invariant:** đáp án luôn nằm trong `[lo, hi]` và `can(hi)` đúng.

**Hướng khác (Union-Find):** xử lý bông theo thứ tự ngày nở; mỗi bông nở ghép với hàng xóm đã nở; số bó tổng = `Σ ⌊len/k⌋`; khi ghép hai đoạn độ dài `a`, `b` với bông mới, số bó thay đổi `⌊(a+b+1)/k⌋ − ⌊a/k⌋ − ⌊b/k⌋`. Ngày đầu tiên tổng ≥ `m` là đáp án.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm kiếm nhị phân trên khoảng ngày `[min, max]`

**Từng đoạn:**
1. Kiểm tra `m * k > n`.
2. `can(day)`: quét một lần, đếm bó với `run` liền kề, **dừng sớm** khi đủ `m`.
3. Nhị phân trên số nguyên `[min(bloomDay), max(bloomDay)]`.

In [10]:
def min_days_binary(bloomDay, m, k):
    if m * k > len(bloomDay):
        return -1

    def can(day):
        count = run = 0
        for b in bloomDay:
            if b <= day:
                run += 1
                if run == k:
                    count += 1
                    if count >= m:
                        return True
                    run = 0
            else:
                run = 0
        return False

    lo, hi = min(bloomDay), max(bloomDay)
    while lo < hi:
        mid = (lo + hi) // 2
        if can(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for b, m, k, want in examples:
    print(min_days_binary(b, m, k), want)

3 3
-1 -1
12 12


Code nộp LeetCode (Cách 2):

In [11]:
class Solution:
    def minDays(self, bloomDay: List[int], m: int, k: int) -> int:
        if m * k > len(bloomDay):
            return -1

        def can(day: int) -> bool:
            count = run = 0
            for b in bloomDay:
                if b <= day:
                    run += 1
                    if run == k:
                        count += 1
                        run = 0
                        if count >= m:
                            return True
                else:
                    run = 0
            return False

        lo, hi = min(bloomDay), max(bloomDay)
        while lo < hi:
            mid = (lo + hi) // 2
            if can(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

print([Solution().minDays(b, m, k) for b, m, k, _ in examples])

[3, -1, 12]


**Độ phức tạp:** `O(n · log(max − min))` ≈ `10^5 × 30 = 3·10^6` bước; bộ nhớ `O(1)` thêm.

### 7.2 Cách 3 — Tìm kiếm nhị phân trên danh sách ngày phân biệt đã sắp

Vì đáp án là một giá trị trong `bloomDay`, nhị phân trên **chỉ số** của `days = sorted(set(bloomDay))`: số vòng `log2(D) ≤ 17` thay vì ~30. Đổi lại tốn `O(n log n)` để sắp.

In [12]:
def min_days_binary_sorted(bloomDay, m, k):
    if m * k > len(bloomDay):
        return -1
    days = sorted(set(bloomDay))
    lo, hi = 0, len(days) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if bouquets_on(bloomDay, k, days[mid]) >= m:
            hi = mid
        else:
            lo = mid + 1
    return days[lo]

for b, m, k, want in examples:
    print(min_days_binary_sorted(b, m, k), want)

3 3
-1 -1
12 12


Code nộp LeetCode (Cách 3):

In [13]:
class Solution:
    def minDays(self, bloomDay: List[int], m: int, k: int) -> int:
        if m * k > len(bloomDay):
            return -1

        def count(day: int) -> int:
            c = run = 0
            for b in bloomDay:
                if b <= day:
                    run += 1
                    if run == k:
                        c += 1
                        run = 0
                else:
                    run = 0
            return c

        days = sorted(set(bloomDay))
        lo, hi = 0, len(days) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if count(days[mid]) >= m:
                hi = mid
            else:
                lo = mid + 1
        return days[lo]

print([Solution().minDays(b, m, k) for b, m, k, _ in examples])

[3, -1, 12]


**Độ phức tạp:** `O(n log n + n log D)`; bộ nhớ `O(D)` cho `days`.

### 7.3 Cách 4 — Xử lý theo thứ tự nở + Union-Find (offline)

**Từng đoạn:**
1. Sắp chỉ số theo `bloomDay` tăng dần.
2. Lần lượt "cho nở" bông `i`: đoạn mới dài 1. Nếu `i−1` đã nở, ghép; nếu `i+1` đã nở, ghép.
3. Duy trì `total = Σ ⌊len/k⌋`: trước khi ghép, trừ đóng góp của các đoạn cũ; sau khi ghép, cộng đóng góp của đoạn mới.
4. Ngay khi `total >= m`, trả `bloomDay[i]`.

Cách này tìm đáp án trong **một lần quét** sau khi sắp; minh họa kỹ thuật "thêm dần phần tử, duy trì thống kê trên đoạn".

In [14]:
def min_days_dsu(bloomDay, m, k):
    n = len(bloomDay)
    if m * k > n:
        return -1
    parent = list(range(n))
    length = [1] * n
    bloomed = [False] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    total = 0
    for i in sorted(range(n), key=bloomDay.__getitem__):
        bloomed[i] = True
        root, cur_len = i, 1
        for j in (i - 1, i + 1):
            if 0 <= j < n and bloomed[j]:
                rj = find(j)
                total -= length[rj] // k           # bỏ đóng góp của đoạn cũ
                parent[rj] = root
                cur_len += length[rj]
        length[root] = cur_len
        total += cur_len // k                      # cộng đóng góp đoạn mới
        if total >= m:
            return bloomDay[i]
    return -1

for b, m, k, want in examples:
    print(min_days_dsu(b, m, k), want)

3 3
-1 -1
12 12


Ghi chú: khi bông `i` nở, gốc đoạn mới đặt tại `i`; các gốc của đoạn trái/phải trỏ về `i`. Code nộp LeetCode (Cách 4):

In [15]:
class Solution:
    def minDays(self, bloomDay: List[int], m: int, k: int) -> int:
        n = len(bloomDay)
        if m * k > n:
            return -1
        parent = list(range(n))
        length = [1] * n
        bloomed = [False] * n

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        total = 0
        for i in sorted(range(n), key=bloomDay.__getitem__):
            bloomed[i] = True
            cur = 1
            for j in (i - 1, i + 1):
                if 0 <= j < n and bloomed[j]:
                    rj = find(j)
                    total -= length[rj] // k
                    parent[rj] = i
                    cur += length[rj]
            length[i] = cur
            total += cur // k
            if total >= m:
                return bloomDay[i]
        return -1

print([Solution().minDays(b, m, k) for b, m, k, _ in examples])

[3, -1, 12]


**Độ phức tạp:** `O(n log n)` (sắp) + `O(n α(n))` (DSU) thời gian; `O(n)` bộ nhớ.

In [16]:
rng = random.Random(5)
big = [rng.randint(1, 10**9) for _ in range(10**5)]
for name, f in [("nhị phân [min,max]", min_days_binary), ("nhị phân ngày sắp", min_days_binary_sorted),
                ("Union-Find", min_days_dsu)]:
    t = time.perf_counter()
    r = f(big, 10**4, 3)
    print(f"{name:19s}: {r}  {time.perf_counter() - t:.3f}s")

nhị phân [min,max] : 572960441  0.160s
nhị phân ngày sắp  : 572960441  0.105s
Union-Find         : 572960441  0.066s


## 8. TRACE từng bước

Trace Cách 2 với `bloomDay = [7,7,7,7,12,7,7]`, `m = 2`, `k = 3` (`lo = 7`, `hi = 12`):

| vòng | lo | hi | mid | số bó ở mid | can? | cập nhật |
|---|---|---|---|---|---|---|
| 1 | 7 | 12 | 9 | 1 | ✘ | lo = 10 |
| 2 | 10 | 12 | 11 | 1 | ✘ | lo = 12 |
| kết thúc | 12 | 12 | | | | đáp án 12 |

In [17]:
def trace_binary(bloomDay, m, k):
    if m * k > len(bloomDay):
        print(f"m*k = {m * k} > n = {len(bloomDay)} -> -1")
        return -1
    lo, hi = min(bloomDay), max(bloomDay)
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        cnt = bouquets_on(bloomDay, k, mid)
        ok = cnt >= m
        garden = "".join("x" if b <= mid else "_" for b in bloomDay)
        print(f"vòng {step}: lo={lo} hi={hi} mid={mid} vườn={garden} số bó={cnt} -> "
              + ("đủ, hi = mid" if ok else "thiếu, lo = mid + 1"))
        if ok:
            hi = mid
        else:
            lo = mid + 1
    print("đáp án:", lo)
    return lo

_ = trace_binary([7, 7, 7, 7, 12, 7, 7], 2, 3)

vòng 1: lo=7 hi=12 mid=9 vườn=xxxx_xx số bó=1 -> thiếu, lo = mid + 1
vòng 2: lo=10 hi=12 mid=11 vườn=xxxx_xx số bó=1 -> thiếu, lo = mid + 1
đáp án: 12


In [18]:
_ = trace_binary([1, 10, 3, 10, 2], 3, 1)
print()
_ = trace_binary([1, 10, 3, 10, 2], 3, 2)

vòng 1: lo=1 hi=10 mid=5 vườn=x_x_x số bó=3 -> đủ, hi = mid
vòng 2: lo=1 hi=5 mid=3 vườn=x_x_x số bó=3 -> đủ, hi = mid
vòng 3: lo=1 hi=3 mid=2 vườn=x___x số bó=2 -> thiếu, lo = mid + 1
đáp án: 3

m*k = 6 > n = 5 -> -1


Trace Cách 4 (Union-Find) trên ví dụ 1 để thấy `total` tăng dần theo thứ tự nở:

In [19]:
def trace_dsu(bloomDay, m, k):
    n = len(bloomDay)
    parent, length, bloomed = list(range(n)), [1] * n, [False] * n
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    total = 0
    for i in sorted(range(n), key=bloomDay.__getitem__):
        bloomed[i] = True
        cur = 1
        for j in (i - 1, i + 1):
            if 0 <= j < n and bloomed[j]:
                rj = find(j)
                total -= length[rj] // k
                parent[rj] = i
                cur += length[rj]
        length[i] = cur
        total += cur // k
        garden = "".join("x" if bloomed[t] else "_" for t in range(n))
        print(f"ngày {bloomDay[i]:2d}: nở bông {i} -> vườn {garden}, đoạn chứa {i} dài {cur}, tổng bó = {total}")
        if total >= m:
            print("đủ m bó -> đáp án", bloomDay[i])
            return bloomDay[i]
    return -1

_ = trace_dsu([1, 10, 3, 10, 2], 3, 1)
print()
_ = trace_dsu([7, 7, 7, 7, 12, 7, 7], 2, 3)

ngày  1: nở bông 0 -> vườn x____, đoạn chứa 0 dài 1, tổng bó = 1
ngày  2: nở bông 4 -> vườn x___x, đoạn chứa 4 dài 1, tổng bó = 2
ngày  3: nở bông 2 -> vườn x_x_x, đoạn chứa 2 dài 1, tổng bó = 3
đủ m bó -> đáp án 3

ngày  7: nở bông 0 -> vườn x______, đoạn chứa 0 dài 1, tổng bó = 0
ngày  7: nở bông 1 -> vườn xx_____, đoạn chứa 1 dài 2, tổng bó = 0
ngày  7: nở bông 2 -> vườn xxx____, đoạn chứa 2 dài 3, tổng bó = 1
ngày  7: nở bông 3 -> vườn xxxx___, đoạn chứa 3 dài 4, tổng bó = 1
ngày  7: nở bông 5 -> vườn xxxx_x_, đoạn chứa 5 dài 1, tổng bó = 1
ngày  7: nở bông 6 -> vườn xxxx_xx, đoạn chứa 6 dài 2, tổng bó = 1
ngày 12: nở bông 4 -> vườn xxxxxxx, đoạn chứa 4 dài 7, tổng bó = 2
đủ m bó -> đáp án 12


## 9. Code nộp LeetCode

Bản khuyên dùng (tìm kiếm nhị phân trên đáp án), đúng chữ ký `minDays(self, bloomDay: List[int], m: int, k: int) -> int`:

In [20]:
class Solution:
    def minDays(self, bloomDay: List[int], m: int, k: int) -> int:
        if m * k > len(bloomDay):            # không đủ bông
            return -1

        def can(day: int) -> bool:           # đến ngày `day` có làm được >= m bó?
            count = run = 0
            for b in bloomDay:
                if b <= day:
                    run += 1
                    if run == k:
                        count += 1
                        run = 0
                        if count >= m:
                            return True
                else:
                    run = 0
            return False

        lo, hi = min(bloomDay), max(bloomDay)
        while lo < hi:
            mid = (lo + hi) // 2
            if can(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

for b, m, k, want in examples:
    assert Solution().minDays(b, m, k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **`m·k > n` ⇒ `-1`.** Mỗi bó cần `k` bông riêng biệt ⇒ cần ít nhất `m·k` bông.
2. **Ngược lại luôn làm được ở ngày `max`.** Mọi bông đã nở, chia vườn thành các khối `k` bông liên tiếp được `⌊n/k⌋ ≥ m` bó.
3. **Đếm tham lam là tối ưu cho một ngày cố định.** Trong một chuỗi liền kề dài `L` bông đã nở, số bó tối đa là `⌊L/k⌋`; quét trái → phải và cắt bó ngay khi đủ `k` đạt đúng giá trị này (cắt sớm không bao giờ làm mất cơ hội — phần còn lại vẫn là một chuỗi liền kề).
4. **Đơn điệu.** Nếu `day1 < day2` thì tập bông nở ở `day1` là tập con của ở `day2`; mỗi chuỗi liền kề ở `day1` nằm trong một chuỗi ở `day2` ⇒ số bó không giảm.
5. **Nhị phân.** Bất biến: `can(hi)` đúng và mọi ngày `< lo` đều sai. Nếu `can(mid)` thì đáp án ≤ `mid` ⇒ `hi = mid`; ngược lại đáp án > `mid` ⇒ `lo = mid + 1`. Khi `lo == hi`, đó là ngày nhỏ nhất thỏa.
6. **Union-Find:** sau khi xử lý mọi bông có `bloomDay ≤ d`, `total` = `Σ ⌊len/k⌋` trên các đoạn đã nở = số bó ở ngày `d` (theo (3)). Duyệt theo ngày tăng, lần đầu `total ≥ m` cho ngày nhỏ nhất. (Các bông cùng ngày được xử lý liên tiếp; kiểm tra sau mỗi bông vẫn đúng vì trả về `bloomDay[i]` = ngày đó.)

## 11. Phân tích Time / Space

Gọi `n = len(bloomDay)`, `V = max − min ≤ 10^9`, `D` số ngày phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng ngày phân biệt | `O(n · D)` = `O(n²)` | `O(D)` |
| 2. Nhị phân trên `[min, max]` | `O(n log V)` ≈ `30n` | `O(1)` |
| 3. Nhị phân trên ngày đã sắp | `O(n log n + n log D)` | `O(D)` |
| 4. Sắp + Union-Find | `O(n log n + n α(n))` | `O(n)` |

Cách 2 nhờ **dừng sớm** trong `can` thường nhanh hơn ước lượng. `m · k` có thể tới `10^6 · 10^5 = 10^11` — Python không tràn số; trong C++/Java cần dùng kiểu 64-bit.

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, `m·k = n`, `m·k > n`, `k = 1`, `k = n`, mọi ngày bằng nhau, `m` rất lớn), đối chiếu brute force trên dữ liệu nhỏ ngẫu nhiên (seed cố định), test hiệu năng `n = 10^5`.

In [21]:
ALL = [min_days_brute, min_days_binary, min_days_binary_sorted, min_days_dsu,
       lambda b, m, k: Solution().minDays(b, m, k)]

def check_all(b, m, k, want=None):
    res = [f(list(b), m, k) for f in ALL]
    if want is None:
        want = res[0]
    assert all(r == want for r in res), (b, m, k, res)
    return want

for b, m, k, want in examples:
    check_all(b, m, k, want)

assert check_all([5], 1, 1) == 5
assert check_all([5], 2, 1) == -1
assert check_all([3, 1, 2], 1, 3) == 3                   # k = n: cần cả vườn
assert check_all([1, 2, 3, 4], 2, 2) == 4                # m*k = n
assert check_all([4, 4, 4, 4], 2, 2) == 4                # mọi ngày bằng nhau
assert check_all([1, 9, 1, 1, 1], 2, 2) == 9             # chuỗi [1,1,1] chỉ cho 1 bó
assert check_all([2] * 10, 10**6, 1) == -1               # m rất lớn
assert check_all([10**9, 1, 10**9], 1, 1) == 1
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [22]:
rng = random.Random(2019)
cnt_neg = 0
for _ in range(3000):
    n = rng.randint(1, 15)
    b = [rng.randint(1, rng.choice([3, 20, 10**9])) for _ in range(n)]
    k = rng.randint(1, n)
    m = rng.randint(1, max(1, n // k + 1))
    cnt_neg += check_all(b, m, k) == -1
print("Random 3000 test: OK, số ca -1 =", cnt_neg)

Random 3000 test: OK, số ca -1 = 1203


In [23]:
# Ràng buộc lớn
rng = random.Random(77)
big = [rng.randint(1, 10**9) for _ in range(10**5)]
for m, k in ((1, 1), (10**4, 5), (33333, 3), (1, 10**5), (10**6, 1)):
    t = time.perf_counter()
    r = Solution().minDays(big, m, k)
    dt = time.perf_counter() - t
    assert r == min_days_dsu(big, m, k)
    print(f"m={m}, k={k}: {r}, {dt:.3f}s")

m=1, k=1: 12226, 0.016s


m=10000, k=5: 807837474, 0.157s


m=33333, k=3: 999998805, 0.181s


m=1, k=100000: 999998805, 0.181s
m=1000000, k=1: -1, 0.000s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra `m · k > n`

Tìm nhị phân vẫn chạy và trả về một ngày (thường là `max`) dù không thể làm đủ bó.

In [24]:
def wrong_no_impossible_check(bloomDay, m, k):
    lo, hi = min(bloomDay), max(bloomDay)         # SAI: thiếu kiểm tra m*k > n
    while lo < hi:
        mid = (lo + hi) // 2
        if bouquets_on(bloomDay, k, mid) >= m:
            hi = mid
        else:
            lo = mid + 1
    return lo

b, m, k, want = examples[1]
print("SAI :", wrong_no_impossible_check(b, m, k), "| đúng:", want)

SAI : 10 | đúng: -1


**Sửa:** `if m * k > len(bloomDay): return -1` trước khi tìm.

### Lỗi 2 — Đếm tổng số bông đã nở chia `k` (bỏ qua "liền kề")

In [25]:
def wrong_not_adjacent(bloomDay, m, k):
    if m * k > len(bloomDay):
        return -1
    for day in sorted(set(bloomDay)):
        if sum(b <= day for b in bloomDay) // k >= m:   # SAI: không xét liền kề
            return day
    return -1

b, m, k, want = examples[2]
print("SAI :", wrong_not_adjacent(b, m, k), "| đúng:", want)

SAI : 7 | đúng: 12


Ngày 7 có 6 bông nở ⇒ `6 // 3 = 2` bó (sai), nhưng thực tế chỉ có chuỗi `4` và `2` ⇒ 1 bó. **Sửa:** đếm theo chuỗi liền kề.

### Lỗi 3 — Không đặt lại `run` sau khi làm một bó

In [26]:
def wrong_no_reset_count(bloomDay, k, day):
    count = run = 0
    for b in bloomDay:
        if b <= day:
            run += 1
            if run >= k:                           # SAI: không reset -> đếm trùng bông
                count += 1
        else:
            run = 0
    return count

def wrong_no_reset(bloomDay, m, k):
    if m * k > len(bloomDay):
        return -1
    for day in sorted(set(bloomDay)):
        if wrong_no_reset_count(bloomDay, k, day) >= m:
            return day
    return -1

b, m, k = [1, 1, 1, 9, 1], 2, 2
print("SAI :", wrong_no_reset(b, m, k), "| đúng:", min_days_binary(b, m, k))

SAI : 1 | đúng: 9


Chuỗi `[1,1,1]` bị đếm thành 2 bó (bông giữa dùng hai lần). **Sửa:** `if run == k: count += 1; run = 0`.

### Lỗi 4 — Cập nhật biên nhị phân sai: `hi = mid − 1`

In [27]:
def wrong_bounds(bloomDay, m, k):
    if m * k > len(bloomDay):
        return -1
    lo, hi = min(bloomDay), max(bloomDay)
    while lo < hi:
        mid = (lo + hi) // 2
        if bouquets_on(bloomDay, k, mid) >= m:
            hi = mid - 1                           # SAI: có thể loại mất đáp án mid
        else:
            lo = mid + 1
    return lo

b, m, k = [9, 1, 7], 2, 1
print("SAI :", wrong_bounds(b, m, k), "| đúng:", min_days_binary(b, m, k),
      "| ngày 6 có đủ không?", bouquets_on(b, k, 6) >= m)

SAI : 6 | đúng: 7 | ngày 6 có đủ không? False


Bản sai trả ngày `6` — ngày mà thực tế **chưa** đủ bó. **Sửa:** khi `can(mid)` đúng, giữ `hi = mid` (mid có thể chính là đáp án).

### Lỗi 5 — "Chữa" trường hợp không thể bằng `lo == max(bloomDay)`

Không kiểm tra `m · k > n` mà sau khi nhị phân lại suy luận "nếu kết quả là `max(bloomDay)` thì chắc là không thể" — sai khi đáp án thật đúng bằng `max`.

In [28]:
def wrong_minus_one_patch(bloomDay, m, k):
    lo, hi = min(bloomDay), max(bloomDay)
    while lo < hi:
        mid = (lo + hi) // 2
        if bouquets_on(bloomDay, k, mid) >= m:
            hi = mid
        else:
            lo = mid + 1
    return -1 if lo == max(bloomDay) else lo      # SAI: nhầm "đáp án = max" với "không thể"

b, m, k, want = examples[2]
print("SAI :", wrong_minus_one_patch(b, m, k), "| đúng:", want)

SAI : -1 | đúng: 12


**Sửa:** phân biệt "không thể" bằng điều kiện `m * k > n` (hoặc kiểm tra `can(max)`), không dựa vào giá trị trả về.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Nhỏ nhất** (hoặc lớn nhất) giá trị `x` sao cho **có thể** làm được …" + kiểm tra một `x` cho trước dễ ⇒ **tìm kiếm nhị phân trên đáp án** (binary search on answer).
- Dấu hiệu đơn điệu: tăng `x` thì "dễ" hơn (thêm thời gian, thêm sức chứa, thêm tốc độ…).
- Họ bài cùng khuôn: 875 Koko Eating Bananas, 1011 Capacity To Ship Packages, 410 Split Array Largest Sum, 1552 Magnetic Force.
- Khuôn nhị phân an toàn: `while lo < hi: mid = (lo+hi)//2; ok → hi = mid; else lo = mid+1`.
- Luôn xử lý trường hợp **không thể** riêng (ở đây `m·k > n`) trước khi nhị phân.
- Đếm khối `k` liền kề: biến `run`, reset khi gặp phần tử xấu **và** khi vừa đủ một khối.

## 15. Biến thể của bài

1. **Bó không cần liền kề:** chỉ cần `số bông đã nở ≥ m·k` ⇒ đáp án là phần tử nhỏ thứ `m·k` (sắp hoặc quickselect).
2. **Số bó tối đa ở ngày `d`** (truy vấn nhiều `d`): xử lý offline theo `d` tăng với Union-Find (Cách 4) — bài tập 5.
3. **Mỗi bó cần đúng `k` bông nhưng được phép bỏ qua tối đa `g` bông chưa nở xen giữa:** thay đổi hàm `can`, khuôn nhị phân giữ nguyên.
4. **875 Koko Eating Bananas / 1011 Ship Packages:** cùng khuôn "nhỏ nhất thỏa điều kiện đơn điệu".
5. **Hoa héo sau `w` ngày** (bông nở ở `[bloomDay, bloomDay + w]`): mất tính đơn điệu ⇒ phải quét các ngày ứng viên hoặc dùng cấu trúc khác.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng ngày | Duyệt ngày phân biệt tăng dần | `O(n²)` | `O(n)` | Kiểm chứng |
| 2. Nhị phân `[min,max]` | Đơn điệu + `can(day)` | `O(n log V)` | `O(1)` | Mặc định, khuôn chuẩn |
| 3. Nhị phân ngày đã sắp | Ít vòng hơn, tốn sort | `O(n log n)` | `O(n)` | Khi `V` rất lớn so với `n` |
| 4. Union-Find offline | Thêm bông theo ngày, duy trì `Σ⌊len/k⌋` | `O(n log n)` | `O(n)` | Cần số bó theo mọi ngày |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minDays([1,10,2,9,3,8,4,7,5,6], 4, 2)` = ?

**Bài 2 (tính tay):** Với bài 1, viết vườn (`x`/`_`) và số bó ở các ngày `5`, `6`, `8`, `9`.

**Bài 3 (điền TODO):** Viết `max_bouquets_on(bloomDay, k, day)` trả về số bó tối đa làm được đúng vào ngày `day`.

**Bài 4 (tự viết lại):** Viết lại khuôn nhị phân dưới dạng hàm tổng quát `first_true_int(lo, hi, ok)` và dùng nó giải bài gốc trong 5 dòng.

**Bài 5 (mini challenge):** Viết `bouquets_by_day(bloomDay, k)` trả về danh sách `(day, số_bó)` cho mọi ngày phân biệt, tổng thời gian `O(n log n)` (gợi ý: Cách 4).

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_bouquets_on(bloomDay: List[int], k: int, day: int) -> int:
    # TODO: biến run, reset khi gặp bông chưa nở và khi đủ k
    raise NotImplementedError

# Bài 4 — TODO
def first_true_int(lo: int, hi: int, ok) -> int:
    # TODO: while lo < hi ...
    raise NotImplementedError

# Bài 5 — TODO
def bouquets_by_day(bloomDay: List[int], k: int):
    # TODO: sắp chỉ số theo ngày, Union-Find, ghi lại total sau khi xử lý hết các bông cùng ngày
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `m·k = 8 ≤ 10`. (2) Thử nhị phân hoặc liệt kê vườn theo ngày. (3) Ngày 9 vườn là `x_xxxxxxxx` (chỉ bông giá trị 10 chưa nở): chuỗi dài 8 cho 4 bó; ngày 8 vườn `x_x_xxxxxx` chỉ cho 3 bó ⇒ đáp án 9.

**Bài 2.** (1) Ngày 5: các bông có giá trị ≤ 5 nở. (2) Đếm chuỗi liền kề và `⌊len/2⌋`. (3) So với kết quả bài 1.

**Bài 3.** (1) Copy `bouquets_on`. (2) `run == k` ⇒ `count += 1`, `run = 0`. (3) Gặp bông chưa nở ⇒ `run = 0`.

**Bài 4.** (1) Khuôn ở mục 3.3. (2) `ok = lambda d: max_bouquets_on(b, k, d) >= m`. (3) Đừng quên `-1` khi `m·k > n`.

**Bài 5.** (1) Sắp chỉ số theo `bloomDay`. (2) Gom các bông cùng ngày, xử lý hết rồi mới ghi `(day, total)`. (3) Đối chiếu với `bouquets_on` cho từng ngày.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, Bài 2
b1 = [1, 10, 2, 9, 3, 8, 4, 7, 5, 6]
r1 = min_days_binary(b1, 4, 2)
print("Bài 1:", r1)
assert r1 == min_days_brute(b1, 4, 2) == 9
for d in (5, 6, 8, 9):
    print(f"Bài 2 — ngày {d}: {''.join('x' if x <= d else '_' for x in b1)} -> {bouquets_on(b1, 2, d)} bó")
assert [bouquets_on(b1, 2, d) for d in (5, 6, 8, 9)] == [0, 1, 3, 4]

Bài 1: 9
Bài 2 — ngày 5: x_x_x_x_x_ -> 0 bó
Bài 2 — ngày 6: x_x_x_x_xx -> 1 bó
Bài 2 — ngày 8: x_x_xxxxxx -> 3 bó
Bài 2 — ngày 9: x_xxxxxxxx -> 4 bó


In [31]:
def max_bouquets_on_ans(bloomDay, k, day):
    count = run = 0
    for b in bloomDay:
        if b <= day:
            run += 1
            if run == k:
                count += 1
                run = 0
        else:
            run = 0
    return count

def first_true_int_ans(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

def min_days_short(b, m, k):
    if m * k > len(b):
        return -1
    return first_true_int_ans(min(b), max(b), lambda d: max_bouquets_on_ans(b, k, d) >= m)

def runs_brute(bloomDay, k, day):
    s = "".join("x" if x <= day else "_" for x in bloomDay)
    return sum(len(seg) // k for seg in s.split("_"))

rng = random.Random(37)
for _ in range(1500):
    b = [rng.randint(1, 12) for _ in range(rng.randint(1, 15))]
    k = rng.randint(1, len(b)); m = rng.randint(1, 5); d = rng.randint(0, 13)
    assert max_bouquets_on_ans(b, k, d) == runs_brute(b, k, d)
    assert min_days_short(b, m, k) == min_days_brute(b, m, k)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [32]:
def bouquets_by_day_ans(bloomDay, k):
    n = len(bloomDay)
    parent, length, bloomed = list(range(n)), [1] * n, [False] * n
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    order = sorted(range(n), key=bloomDay.__getitem__)
    total, out, idx = 0, [], 0
    while idx < n:
        day = bloomDay[order[idx]]
        while idx < n and bloomDay[order[idx]] == day:     # xử lý hết các bông cùng ngày
            i = order[idx]
            bloomed[i] = True
            cur = 1
            for j in (i - 1, i + 1):
                if 0 <= j < n and bloomed[j]:
                    rj = find(j)
                    total -= length[rj] // k
                    parent[rj] = i
                    cur += length[rj]
            length[i] = cur
            total += cur // k
            idx += 1
        out.append((day, total))
    return out

rng = random.Random(43)
for _ in range(500):
    b = [rng.randint(1, 10) for _ in range(rng.randint(1, 20))]
    k = rng.randint(1, 4)
    assert bouquets_by_day_ans(b, k) == [(d, runs_brute(b, k, d)) for d in sorted(set(b))]
print("Bài 5:", bouquets_by_day_ans([7, 7, 7, 7, 12, 7, 7], 3), "| OK 500 test")

Bài 5: [(7, 1), (12, 2)] | OK 500 test


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "binary search on answer":**

```python
if m * k > len(bloomDay):
    return -1                                  # trường hợp không thể

def can(day):
    count = run = 0
    for b in bloomDay:
        if b <= day:
            run += 1
            if run == k:
                count += 1; run = 0
                if count >= m: return True
        else:
            run = 0
    return False

lo, hi = min(bloomDay), max(bloomDay)
while lo < hi:
    mid = (lo + hi) // 2
    if can(mid): hi = mid
    else:        lo = mid + 1
return lo
```

**Checklist:**
- [ ] Kiểm tra `m * k > n` ⇒ `-1` trước.
- [ ] Đếm bó theo **chuỗi liền kề**, reset `run` khi gặp bông chưa nở **và** khi đủ `k`.
- [ ] Nhị phân: `hi = mid` khi đúng, `lo = mid + 1` khi sai; vòng `while lo < hi`.
- [ ] Khoảng tìm `[min, max]` của `bloomDay` (hoặc danh sách ngày phân biệt).
- [ ] Đã thử: `n = 1`, `k = n`, `m·k = n`, mọi ngày bằng nhau, đáp án đúng bằng `max`.